In [ ]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
from typing import List
import time
import os
import numpy as np
import time
import os
import numpy as np
import torch
import pickle
import argparse
from uuid import uuid4

from torch.utils.data import DataLoader

import sys
sys.path.append('..')
sys.path.append('../..')
sys.path.append('../../..')
from reactot.trainer.pl_trainer import SBModule
from reactot.dataset.transition1x import ProcessedTS1x
from reactot.analyze.rmsd import batch_rmsd
from reactot.analyze.geomopt import calc_deltaE, compute_efh
from reactot.evaluate.utils import (
    set_new_schedule,
    inplaint_batch,
    batch_ts_deltaE,
)
from reactot.utils.sampling_tools import write_tmp_xyz

EV2KCALMOL = 23.06
AU2KCALMOL = 627.5

In [ ]:
from Model.backbone import generate_backbone
from Model.head import generate_head
from Model.model import MDNet

import yaml
from easydict import EasyDict

from ase import Atoms

In [ ]:
import numpy as np
from ase.calculators.calculator import (Calculator, CalculatorError, 
                    CalculatorSetupError, all_changes, all_properties, kpts2mp, FileIOCalculator)
from pyscf import gto, dft
from pyscf.geomopt.geometric_solver import optimize
from pyscf.hessian import thermo
import pyscf

AU2KCALMOL = 627.509608
AU2EV = 27.2114
BOHR = 0.52917721092

def ase_atoms_to_pyscf(ase_atoms):
    '''Convert ASE atoms to PySCF atom.

    Note: ASE atoms always use A.
    '''
    return [[atom.symbol, atom.position] for atom in ase_atoms]

atoms_from_ase = ase_atoms_to_pyscf

def calculate_efh(
    atom,
    f=True,
    hess=False,
    return_metrics=False,
    xc="wb97x",
    basis="631g*",
    device='gpu',
    # d3=False,
):
    geomfile = ase_atoms_to_pyscf(atom)
    spin = 0
    mol = pyscf.M(
        atom=geomfile,
        unit="Ang",
        basis=basis,
    )
    mol.build()

    if device == 'gpu':
        mf = dft.RKS(mol).to_gpu() if not spin else dft.UKS(mol).to_gpu()
    else:
        mf = dft.RKS(mol) if not spin else dft.UKS(mol)
    mf.xc = xc
    # if d3:
    #     mf = dftd3.dftd3(dft.RKS(mol, xc=xc))
    mf.conv_tol = 1e-6
    # mf.damp = 0.2
    mf.max_cycle = 200
    mf.max_memory = 32000
    mf.run()

    force = None
    force_rms = np.nan
    if mf.converged and f:
        force = mf.nuc_grad_method().kernel() * -1.0 / BOHR
        force_rms = np.sqrt(np.mean(force**2)) * AU2EV
        print("force rms (ev/A): ", force_rms)

    hessian = None
    if hess:
        hessian = mf.Hessian().kernel()
        freq_info = thermo.harmonic_analysis(mf.mol, hessian)
        print("freq: ", freq_info["freq_wavenumber"])

    if return_metrics:
        return (
            mf,
            force,
            hessian,
            force_rms,
            count_negative_eig(freq_info["freq_wavenumber"]),
        )
    return mf, force, hessian

# K-point sampling
def make_kpts(cell, nks):
    raise DeprecationWarning('Use cell.make_kpts(nks) instead.')

def count_negative_eig(x: list):
    count = 0
    for _x in x:
        if _x.imag > 0:
            count += 1
    return count

In [ ]:
parser = argparse.ArgumentParser(description='Training Transition1x dynamics')
parser.add_argument('--config_file', required=True)
parser.add_argument('--log_prefix', default='logs')
parser.add_argument('--notes', default=' ')
parser.add_argument('--device', default='cuda')
parser.add_argument('--resume_status', default=' ')
parser.add_argument('--potential', default=' ')
args = parser.parse_args(['--config_file', "../../Configs/Potential.yml",
                          '--device', 'cuda',
                          '--potential', '/home/yufeiluo/research/transition_state_pred/logs/potential/no_pretrained_model_2025_08_13__21_54_34/checkpoints/checkpoint_best.pth'])

In [ ]:
def load_model(
    checkpoint_path,
):
    print (checkpoint_path)
    model = SBModule.load_from_checkpoint(
        checkpoint_path=checkpoint_path,
        map_location=args.device,
    )
    model = model.eval()
    model = model.to(args.device)

    model.training_config["use_sampler"] = False
    model.training_config["swapping_react_prod"] = False
    model.training_config["datadir"] = "./data/transition1x"

    model.setup(stage="test", device=args.device, swapping_react_prod=False)
    return model

opt = {
    "batch_size": 1,
    "nfe": 10,
    "solver": "ode",
    "checkpoint": "", # path to trained React-OT checkpoint
    "order": 1,
    "diz": "linear",
    "method": "midpoint",
    "atol": 1e-2,
    "rtol": 1e-2
}
opt = EasyDict(opt)

In [ ]:
dtype = torch.float32

config_path=args.config_file
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
config = EasyDict(config)
config.notes = args.notes

device = args.device
backbone = generate_backbone(config.model.backbone)
head = generate_head(config.model.head)

In [ ]:
REFERENCE_ENERGIES = {
    1: -13.62222753701504,
    6: -1029.4130839658328,
    7: -1484.8710358098756,
    8: -2041.8396277138045,
    9: -2712.8213146878606,
}

In [ ]:
potential_model = MDNet(backbone, head, REFERENCE_ENERGIES)
best_state = torch.load(args.potential, map_location=device)
potential_model.load_state_dict(best_state['model'])
potential_model.eval()

In [ ]:
potential_model = potential_model.to(device)

In [ ]:
model = load_model(opt.checkpoint)

test_loader = model.test_dataloader(bz=opt.batch_size)
# test_loader = model.train_dataloader(bz=opt.batch_size)
model.nfe = opt.nfe
model.ddpm.opt = opt # hack :)

In [ ]:
rmsds_list, dmaes_list, deltaEs_list = [], [], []
deltaEs_dft_list = []
reactant_product_pos = []
transition_state_pos_true = []
transition_state_pos_pred = []
atom_types = []
true_energy_barrier = []
for ii, batch in enumerate(test_loader):
    pred_transition_state_pos, reps, conds, ediff, rmsds, dmaes = model.eval_sample_batch(
    batch, test=True
    )  # 30s for nfe=100

    transition_state_pos_pred.append(pred_transition_state_pos)
    

    true_energy_barrier.append(ediff)
    
    atom_type = reps[0]['charge'].type(torch.long).squeeze().to(device)
    reactant_pos = reps[0]['pos'].to(device)
    product_pos = reps[2]['pos']
    trans_pos = reps[1]['pos']
    transition_state_pos_true.append(trans_pos)

    reactant_product_pos.append((reactant_pos.cpu().numpy(), product_pos.cpu().numpy()))
    atom_types.append(atom_type.cpu().numpy())

    batch_num = torch.zeros_like(atom_type)
    pred_transition_state_pos = pred_transition_state_pos.to(device)

    atom_reactant = Atoms(numbers=atom_type.cpu().numpy(), positions=reactant_pos.detach().cpu().numpy())
    atom_trans = Atoms(numbers=atom_type.cpu().numpy(), positions=pred_transition_state_pos.detach().cpu().numpy())

    # energy_dft_react = calculate_efh(atom_reactant, f=True)[0].e_tot * AU2EV
    # energy_dft_trans = calculate_efh(atom_trans, f=True)[0].e_tot * AU2EV
    # energy_barrier_dft = energy_dft_react - energy_dft_trans
    energy_barrier_dft = 0.0

    energy_pred_react, _ = potential_model.get_energy_and_force(atom_type, reactant_pos, None, None, batch_num)
    energy_pred_trans, __ = potential_model.get_energy_and_force(atom_type, pred_transition_state_pos, None, None, batch_num)

    energy_barrier_pred = (energy_pred_react - energy_pred_trans).detach().cpu()
    _deltaEs = torch.abs(energy_barrier_pred - ediff).numpy()

    _deltaEs_dft = np.abs(energy_barrier_dft - ediff)

    rmsds_list.append(rmsds)
    dmaes_list.append(dmaes)
    deltaEs_list.append(_deltaEs)
    deltaEs_dft_list.append(_deltaEs_dft)

In [ ]:
import pickle
with open('res_reactot.pickle', 'wb') as f:
    pickle.dump({
        'rmsd': rmsds_list,
        'dmae': dmaes_list,
        # 'barriers_model': deltaEs_list,
        # 'barriers_dft': deltaEs_dft_list
        'reactant_product_pos': reactant_product_pos,
        'atom_types': atom_types,
        'true_energy_barrier_reactant': true_energy_barrier,
        'true_transition_state_pos': transition_state_pos_true,
        'pred_transition_state_pos': transition_state_pos_pred
    }, f)